In [1]:
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import datetime
from itertools import chain
import os
import tqdm
import glob

In [2]:
os.getcwd()

'/home/mds_backfill'

In [3]:
os.chdir('/home/mds_backfill/rpc_feature_extraction/rpc_model/sample_data_individual_feat')

FileNotFoundError: [Errno 2] No such file or directory: '/home/mds_backfill/rpc_feature_extraction/rpc_model/sample_data_individual_feat'

In [9]:
files = glob.glob('*.{}'.format('csv'))
files.sort()
files

['bounce_item_w1_2022-10-16.csv',
 'bounce_item_w1_2022-10-23.csv',
 'bounce_item_w1_2022-10-30.csv',
 'bounce_item_w1_2022-11-06.csv',
 'bounce_item_w1_2022-11-13.csv',
 'bounce_item_w1_2022-11-20.csv',
 'bounce_item_w2_2022-10-16.csv',
 'bounce_item_w2_2022-10-23.csv',
 'bounce_item_w2_2022-10-30.csv',
 'bounce_item_w2_2022-11-06.csv',
 'bounce_item_w2_2022-11-13.csv',
 'bounce_item_w2_2022-11-20.csv',
 'catalog_w1_2022-10-16.csv',
 'catalog_w1_2022-10-23.csv',
 'catalog_w1_2022-10-30.csv',
 'catalog_w1_2022-11-06.csv',
 'catalog_w1_2022-11-13.csv',
 'catalog_w1_2022-11-20.csv',
 'df_header_2022-10-16.csv',
 'df_header_2022-10-23.csv',
 'df_header_2022-10-30.csv',
 'df_header_2022-11-06.csv',
 'df_header_2022-11-13.csv',
 'df_header_2022-11-20.csv',
 'semcoop_l_2022-10-16.csv',
 'semcoop_l_2022-10-23.csv',
 'semcoop_l_2022-10-30.csv',
 'semcoop_l_2022-11-06.csv',
 'semcoop_l_2022-11-13.csv',
 'semcoop_l_2022-11-20.csv',
 'semcoop_v12_2022-10-16.csv',
 'semcoop_v12_2022-10-23.csv',
 '

In [9]:
aa.head()

,Unnamed: 0,catalog_item_id,seller_id,adid,source_id,is_mobile,semcoop_impressions_l,semcoop_clicks_l,semcoop_adspend_l,semcoop_qty_l,...,bc_sem_bounce_rate_w1,bc_sem_bounces_w1,bc_visits_w1,bc_sem_visits_w1,bc_bounce_rate_w2,bc_bounces_w2,bc_sem_bounce_rate_w2,bc_sem_bounces_w2,bc_visits_w2,bc_sem_visits_w2
0,0,1000438425,101197600,1000438425_141830104220_18285736203,2,0,NaN,NaN,NaN,NaN,...,0.680000,17.0,1285.0,25.0,0.720934,4291.0,0.75,45.0,5952.0,60.0
1,1,1000641068,101137322,1000641068_101137322_140940758340_18064667719,2,0,NaN,NaN,NaN,NaN,...,NaN,0.0,1.0,0.0,0.000000,0.0,NaN,0.0,2.0,0.0
2,2,1000956959,101126726,1000956959_144893864830_18291203882,2,1,NaN,NaN,NaN,NaN,...,NaN,0.0,51.0,0.0,0.231788,35.0,0.00,0.0,151.0,1.0
3,3,1001819367,0,1001819367_141412546465_18497457021,2,0,0.0,0.0,0.0,-0.142857,...,0.666667,2.0,162.0,3.0,0.339901,138.0,0.50,2.0,406.0,4.0
4,4,1001852866,101135396,1001852866_141989961998_18286110951,2,0,NaN,NaN,NaN,NaN,...,NaN,0.0,4.0,0.0,0.625000,5.0,NaN,0.0,8.0,0.0


In [5]:
import pandas as pd
start_date = '2022-10-16'
end_date = '2022-11-20'
ref_dates = pd.date_range(start=pd.to_datetime(start_date), end=pd.to_datetime(end_date), freq='7D')
ref_dates = [x.strftime('%Y-%m-%d') for x in ref_dates]

In [6]:
ref_dates

['2022-10-16',
 '2022-10-23',
 '2022-10-30',
 '2022-11-06',
 '2022-11-13',
 '2022-11-20']

In [10]:
for date in ref_dates:
    print('working on date '+date)
    # read header
    file = 'df_header_'+date+'.csv'
    df_feat_all = pd.read_csv(file, index_col=0)
    
    df_feat_all = df_feat_all[(~pd.isnull(df_feat_all['catalog_item_id']))&(~pd.isnull(df_feat_all['seller_id']))]
    df_feat_all['catalog_item_id'] = df_feat_all['catalog_item_id'].astype('int')
    df_feat_all['catalog_item_id'] = df_feat_all['catalog_item_id'].astype('str')
    df_feat_all['seller_id'] = df_feat_all['seller_id'].astype('int')
    df_feat_all['seller_id'] = df_feat_all['seller_id'].astype('str')
    df_feat_all.drop_duplicates(subset = 'adid', inplace=True)
    
    # drop off duplicates catalog_item_id, seller_id
    
#     # read targets
#     file = 'sem_l_'+date+'.csv'
#     df_tmp = pd.read_csv(file, index_col=0)
#     df_feat_all = df_tmp.merge(df_feat_all, left_on = 'catalog_item_id', right_on = 'catalog_item_id', how='left')
    
    # read sem coop features
    files_to_read = [x for x in files if x[-14:-4] == date and x[:4]=='semc']
    
    # left join to ensure item coverage is based on target coverage
    for file in files_to_read:
        print('merge sem coop features')
        df_tmp = pd.read_csv(file, index_col=0)
        df_tmp = df_tmp[(~pd.isnull(df_tmp['catalog_item_id']))&(~pd.isnull(df_tmp['seller_id']))]
        df_tmp['catalog_item_id'] = df_tmp['catalog_item_id'].astype('int')
        df_tmp['catalog_item_id'] = df_tmp['catalog_item_id'].astype('str')
        df_tmp['seller_id'] = df_tmp['seller_id'].astype('int')
        df_tmp['seller_id'] = df_tmp['seller_id'].astype('str')
        df_tmp.drop_duplicates(subset = 'adid', inplace=True)
        df_feat_all = df_feat_all.merge(df_tmp, on=['catalog_item_id', 'seller_id', 'source_id', 'adid', 'is_mobile'], how='left')
        del df_tmp
        
    # read catalog feature
    files_to_read = [x for x in files if x[-14:-4] == date and x[:7]=='catalog']
    for file in files_to_read:
        print('merge catalog features')
        df_tmp = pd.read_csv(file, index_col=0)
        df_tmp = df_tmp[(~pd.isnull(df_tmp['catalog_item_id']))&(~pd.isnull(df_tmp['seller_id']))]
        df_tmp['catalog_item_id'] = df_tmp['catalog_item_id'].astype('int')
        df_tmp['catalog_item_id'] = df_tmp['catalog_item_id'].astype('str')
        df_tmp['seller_id'] = df_tmp['seller_id'].astype('int')
        df_tmp['seller_id'] = df_tmp['seller_id'].astype('str')
        df_tmp.drop_duplicates(subset = ['catalog_item_id','seller_id'], inplace=True)
        df_feat_all = df_feat_all.merge(df_tmp, on=['catalog_item_id', 'seller_id'], how='left')
        del df_tmp
        
    # read site features
    files_to_read = [x for x in files if x[-14:-4] == date and x[:4]=='site' and 'item' in x]
    for file in files_to_read:
        print('merge site features')
        df_tmp = pd.read_csv(file, index_col=0)
        df_tmp = df_tmp[(~pd.isnull(df_tmp['catalog_item_id']))&(~pd.isnull(df_tmp['seller_id']))]
        df_tmp['catalog_item_id'] = df_tmp['catalog_item_id'].astype('int')
        df_tmp['catalog_item_id'] = df_tmp['catalog_item_id'].astype('str')
        df_tmp['seller_id'] = df_tmp['seller_id'].astype('int')
        df_tmp['seller_id'] = df_tmp['seller_id'].astype('str')
        df_tmp.drop_duplicates(subset = ['catalog_item_id','seller_id'], inplace=True)
        df_feat_all = df_feat_all.merge(df_tmp, on=['catalog_item_id', 'seller_id'], how='left')
        del df_tmp
        
    # read bounce features
    files_to_read = [x for x in files if x[-14:-4] == date and x[:6]=='bounce' and 'item' in x]
    for file in files_to_read:
        print('merge bounce features')
        df_tmp = pd.read_csv(file, index_col=0)
        df_tmp = df_tmp[(~pd.isnull(df_tmp['catalog_item_id']))&(~pd.isnull(df_tmp['seller_id']))]
        df_tmp['catalog_item_id'] = df_tmp['catalog_item_id'].astype('int')
        df_tmp['catalog_item_id'] = df_tmp['catalog_item_id'].astype('str')
        df_tmp['seller_id'] = df_tmp['seller_id'].astype('int')
        df_tmp['seller_id'] = df_tmp['seller_id'].astype('str')
        df_tmp.drop_duplicates(subset = ['catalog_item_id','seller_id'], inplace=True)
        df_feat_all = df_feat_all.merge(df_tmp, on=['catalog_item_id', 'seller_id'], how='left')
        del df_tmp
        
    df_feat_all.to_csv('../sample_data_full_feat/df_feat_all_coop_item_'+date+'.csv')
    del df_feat_all

working on date 2022-10-16
merge sem coop features


/opt/conda/anaconda/lib/python3.6/site-packages/IPython/core/interactiveshell.py:3072: DtypeWarning: Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.
  interactivity=interactivity, compiler=compiler, result=result)


merge sem coop features
merge sem coop features
merge sem coop features
merge sem coop features
merge sem coop features
merge sem coop features
merge sem coop features
merge sem coop features
merge sem coop features
merge catalog features
merge site features
merge site features
merge bounce features
merge bounce features
working on date 2022-10-23
merge sem coop features
merge sem coop features
merge sem coop features
merge sem coop features
merge sem coop features
merge sem coop features
merge sem coop features
merge sem coop features
merge sem coop features
merge sem coop features
merge catalog features
merge site features
merge site features
merge bounce features
merge bounce features
working on date 2022-10-30
merge sem coop features
merge sem coop features
merge sem coop features
merge sem coop features
merge sem coop features
merge sem coop features
merge sem coop features
merge sem coop features
merge sem coop features
merge sem coop features
merge catalog features


/opt/conda/anaconda/lib/python3.6/site-packages/IPython/core/interactiveshell.py:3072: DtypeWarning: Columns (1,3,4,5,6,7,8,10) have mixed types. Specify dtype option on import or set low_memory=False.
  interactivity=interactivity, compiler=compiler, result=result)


merge site features
merge site features
merge bounce features
merge bounce features
working on date 2022-11-06


/home/mds_backfill/.local/lib/python3.6/site-packages/numpy/lib/arraysetops.py:580: FutureWarning: elementwise comparison failed; returning scalar instead, but in the future will perform elementwise comparison
  mask |= (ar1 == a)


merge sem coop features
merge sem coop features
merge sem coop features
merge sem coop features
merge sem coop features
merge sem coop features
merge sem coop features
merge sem coop features
merge sem coop features
merge sem coop features
merge catalog features
merge site features
merge site features
merge bounce features
merge bounce features
working on date 2022-11-13
merge sem coop features
merge sem coop features
merge sem coop features
merge sem coop features
merge sem coop features
merge sem coop features
merge sem coop features
merge sem coop features
merge sem coop features
merge sem coop features
merge catalog features
merge site features
merge site features
merge bounce features
merge bounce features
working on date 2022-11-20
merge sem coop features
merge sem coop features
merge sem coop features
merge sem coop features
merge sem coop features
merge sem coop features
merge sem coop features
merge sem coop features
merge sem coop features
merge sem coop features
merge catalo

In [16]:
for date in ref_dates[2:][::-1]:
    print('working on date '+date)
    # read header
    file = 'df_header_'+date+'.csv'
    df_feat_all = pd.read_csv(file, index_col=0)
    
    df_feat_all = df_feat_all[(~pd.isnull(df_feat_all['catalog_item_id']))&(~pd.isnull(df_feat_all['seller_id']))]
    df_feat_all['catalog_item_id'] = df_feat_all['catalog_item_id'].astype('int')
    df_feat_all['catalog_item_id'] = df_feat_all['catalog_item_id'].astype('str')
    df_feat_all['seller_id'] = df_feat_all['seller_id'].astype('int')
    df_feat_all['seller_id'] = df_feat_all['seller_id'].astype('str')
    df_feat_all.drop_duplicates(subset = 'adid', inplace=True)
    
    # drop off duplicates catalog_item_id, seller_id
    
#     # read targets
#     file = 'sem_l_'+date+'.csv'
#     df_tmp = pd.read_csv(file, index_col=0)
#     df_feat_all = df_tmp.merge(df_feat_all, left_on = 'catalog_item_id', right_on = 'catalog_item_id', how='left')
    
    # read sem features
    files_to_read = [x for x in files if x[-14:-4] == date and x[:4]=='sem_']
    
    # left join to ensure item coverage is based on target coverage
    for file in files_to_read:
        print('merge sem features')
        df_tmp = pd.read_csv(file, index_col=0)
        df_tmp = df_tmp[(~pd.isnull(df_tmp['catalog_item_id']))&(~pd.isnull(df_tmp['seller_id']))]
        df_tmp['catalog_item_id'] = df_tmp['catalog_item_id'].astype('int')
        df_tmp['catalog_item_id'] = df_tmp['catalog_item_id'].astype('str')
        df_tmp['seller_id'] = df_tmp['seller_id'].astype('int')
        df_tmp['seller_id'] = df_tmp['seller_id'].astype('str')
        df_tmp.drop_duplicates(subset = 'adid', inplace=True)
        df_feat_all = df_feat_all.merge(df_tmp, on=['catalog_item_id', 'seller_id', 'source_id', 'adid', 'is_mobile'], how='left')
        del df_tmp
        
    # read catalog feature
    files_to_read = [x for x in files if x[-14:-4] == date and x[:7]=='catalog']
    for file in files_to_read:
        print('merge catalog features')
        df_tmp = pd.read_csv(file, index_col=0)
        df_tmp = df_tmp[(~pd.isnull(df_tmp['catalog_item_id']))&(~pd.isnull(df_tmp['seller_id']))]
        df_tmp['catalog_item_id'] = df_tmp['catalog_item_id'].astype('int')
        df_tmp['catalog_item_id'] = df_tmp['catalog_item_id'].astype('str')
        df_tmp['seller_id'] = df_tmp['seller_id'].astype('int')
        df_tmp['seller_id'] = df_tmp['seller_id'].astype('str')
        df_tmp.drop_duplicates(subset = ['catalog_item_id','seller_id'], inplace=True)
        df_feat_all = df_feat_all.merge(df_tmp, on=['catalog_item_id', 'seller_id'], how='left')
        del df_tmp
        
    # read site features
    files_to_read = [x for x in files if x[-14:-4] == date and x[:4]=='site' and 'item' in x]
    for file in files_to_read:
        print('merge site features')
        df_tmp = pd.read_csv(file, index_col=0)
        df_tmp = df_tmp[(~pd.isnull(df_tmp['catalog_item_id']))&(~pd.isnull(df_tmp['seller_id']))]
        df_tmp['catalog_item_id'] = df_tmp['catalog_item_id'].astype('int')
        df_tmp['catalog_item_id'] = df_tmp['catalog_item_id'].astype('str')
        df_tmp['seller_id'] = df_tmp['seller_id'].astype('int')
        df_tmp['seller_id'] = df_tmp['seller_id'].astype('str')
        df_tmp.drop_duplicates(subset = ['catalog_item_id','seller_id'], inplace=True)
        df_feat_all = df_feat_all.merge(df_tmp, on=['catalog_item_id', 'seller_id'], how='left')
        del df_tmp
        
    # read bounce features
    files_to_read = [x for x in files if x[-14:-4] == date and x[:6]=='bounce' and 'item' in x]
    for file in files_to_read:
        print('merge bounce features')
        df_tmp = pd.read_csv(file, index_col=0)
        df_tmp = df_tmp[(~pd.isnull(df_tmp['catalog_item_id']))&(~pd.isnull(df_tmp['seller_id']))]
        df_tmp['catalog_item_id'] = df_tmp['catalog_item_id'].astype('int')
        df_tmp['catalog_item_id'] = df_tmp['catalog_item_id'].astype('str')
        df_tmp['seller_id'] = df_tmp['seller_id'].astype('int')
        df_tmp['seller_id'] = df_tmp['seller_id'].astype('str')
        df_tmp.drop_duplicates(subset = ['catalog_item_id','seller_id'], inplace=True)
        df_feat_all = df_feat_all.merge(df_tmp, on=['catalog_item_id', 'seller_id'], how='left')
        del df_tmp
        
    df_feat_all.to_csv('../sample_data_full_feat/df_feat_all_item_'+date+'.csv')
    del df_feat_all

working on date 2022-11-20
merge sem features
merge sem features
merge sem features
merge sem features
merge sem features
merge sem features
merge sem features
merge sem features
merge sem features
merge sem features
merge catalog features
merge site features
merge site features
merge bounce features
merge bounce features
working on date 2022-11-13
merge sem features
merge sem features
merge sem features
merge sem features
merge sem features
merge sem features
merge sem features
merge sem features
merge sem features
merge sem features
merge catalog features
merge site features
merge site features
merge bounce features
merge bounce features
working on date 2022-11-06
merge sem features
merge sem features
merge sem features
merge sem features
merge sem features
merge sem features
merge sem features
merge sem features
merge sem features
merge sem features
merge catalog features
merge site features
merge site features
merge bounce features
merge bounce features
working on date 2022-10-30
m